In [ ]:
# MCC - Temas Selectos III 
# Instructor: Dr. J.A. Garcia-Rodriguez

# Imports the main libraries and defines general settings for reproducibility,
# data display, warnings, and time-series data handling.
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["NIXTLA_ID_AS_COL"] = "true"

import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)

import random
random.seed(1)

import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)


In [2]:
# Imports a utility function for visualizing time-series data.
from utilsforecast.plotting import plot_series as plot_series_utils

In [ ]:
#Install Seaborn, a Python library for statistical data visualization.
%pip install seaborn

In [ ]:
# Install fpppy, a package with tools for time-series analysis and forecasting.
%pip install fpppy

In [3]:
# Configures Matplotlib and Seaborn for time-series visualization and
# defines custom color palettes and colormaps for consistent plotting.

import seaborn as sns
sns.set_style("whitegrid")

import matplotlib.pyplot as plt
plt.style.use("ggplot")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "grid.alpha": 1.0,
})

import matplotlib as mpl

from cycler import cycler
mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])

from fpppy.utils import plot_series
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#2f2fff"], name="black_and_blue"),
    force=True,
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00"], name="black_and_orange"),
    force=True,
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#000000"], name="black"),
    force=True,
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#0072B2", "#D55E00"],
        name='black_and_2color',
    ),
    force=True
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00", "#0072B2", "#009E73"],
        name='black_and_3color',
    ),
    force=True
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00", "#0072B2", "#009E73", "#CC79A7"],
        name='black_and_4color',
    ),
    force=True
)

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#D55E00", "#0072B2", "#009E73", "#CC79A7"],
        name='r_colors',
    ),
    force=True
)

In [4]:
import statsmodels.api as sm
from scipy.stats import pearsonr
from statsmodels.graphics.tsaplots import plot_acf

In [5]:
df = pd.DataFrame({
    "Year": list(range(2015, 2020)),
    "Observation": [123, 39, 78, 52, 110],
})
df

,Year,Observation
0,2015,123
1,2016,39
2,2017,78
3,2018,52
4,2019,110


In [9]:
print(df.index)
print(df.columns)

RangeIndex(start=0, stop=5, step=1)
Index(['Year', 'Observation'], dtype='object')


In [10]:
df.dtypes

Year           int64
Observation    int64
dtype: object

In [12]:
print(type(df["Year"]))
print(df["Year"])

<class 'pandas.core.series.Series'>
0    2015
1    2016
2    2017
3    2018
4    2019
Name: Year, dtype: int64


In [13]:
year_df = df.set_index("Year")
year_df

,Observation
Year,
2015,123
2016,39
2017,78
2018,52
2019,110


In [14]:
# Time series observations may be associated with instants in time or spans of time. 
# Pandas supports these concepts through pd.Timestamp and pd.Period classes.

print(repr(pd.Timestamp("2020-01")))
print(repr(pd.Period("2020-01")))

Timestamp('2020-01-01 00:00:00')
Period('2020-01', 'M')


In [ ]:
# Pandas provides many utilities for converting between text strings 
# and dedicated timestamp or period representations.

print(repr(pd.Timestamp("2020")))
print(repr(pd.Timestamp("2020-01-01 12:34")))
print(repr(pd.Period("2020-01-01")))
print(repr(pd.Period("2020-01-01", freq="M")))

Timestamp('2020-01-01 00:00:00')
Timestamp('2020-01-01 12:34:00')
Period('2020-01-01', 'D')
Period('2020-01', 'M')


In [28]:
# Timestamp sequences can be conveniently constructed using
# pd.to_datetime() or pd.date_range()

ts_few = pd.to_datetime(["2020-01-01", "2020-01-02", "2020-01-03"])
ts_range = pd.date_range("2020-01-01", "2020-01-02", freq="8h")
print(ts_few)
print(ts_range)

DatetimeIndex(['2020-01-01', '2020-01-02', '2020-01-03'], dtype='datetime64[ns]', freq=None)
DatetimeIndex(['2020-01-01 00:00:00', '2020-01-01 08:00:00',
               '2020-01-01 16:00:00', '2020-01-02 00:00:00'],
              dtype='datetime64[ns]', freq='8h')


In [29]:
# The .to_period() and .to_timestamp() methods allow conversions
# between timestamp and period. 
# The .to_period() method will infer the period duration unless the freq= argument is specified.

print(ts_range.to_period())
print(ts_range.to_period(freq="D"))
print(ts_range.to_period(freq="W"))
print(ts_range.to_period().to_timestamp())

PeriodIndex(['2020-01-01 00:00', '2020-01-01 08:00', '2020-01-01 16:00',
             '2020-01-02 00:00'],
            dtype='period[8h]')
PeriodIndex(['2020-01-01', '2020-01-01', '2020-01-01', '2020-01-02'], dtype='period[D]')
PeriodIndex(['2019-12-30/2020-01-05', '2019-12-30/2020-01-05',
             '2019-12-30/2020-01-05', '2019-12-30/2020-01-05'],
            dtype='period[W-SUN]')
DatetimeIndex(['2020-01-01 00:00:00', '2020-01-01 08:00:00',
               '2020-01-01 16:00:00', '2020-01-02 00:00:00'],
              dtype='datetime64[ns]', freq='8h')


In [ ]:
#Both timestamps and period start times can be converted to strings with custom formatting
print(ts_few.strftime("%m/%d/%Y"))
print(ts_range.to_period().strftime("%Y %b ~ %H:%M"))

Index(['01/01/2020', '01/02/2020', '01/03/2020'], dtype='object')
Index(['2020 Jan ~ 00:00', '2020 Jan ~ 08:00', '2020 Jan ~ 16:00',
       '2020 Jan ~ 00:00'],
      dtype='object')


In [31]:
# Pandas automatically converts between Index and Series types 
# as needed when these sequences are used as the data or index in a DataFrame.

df = pd.DataFrame({"ts": ts_few})
df = df.assign(
    period=df["ts"].dt.to_period(),
    yr=df["ts"].dt.year,
    str=df["ts"].dt.strftime("%A, %B %#d")).set_index("ts")
df

,period,yr,str
ts,,,
2020-01-01,2020-01-01,2020,"Wednesday, January 1"
2020-01-02,2020-01-02,2020,"Thursday, January 2"
2020-01-03,2020-01-03,2020,"Friday, January 3"
